# Structure Representation for Proteins

This notebook covers numerical representations of protein 3D structures.

**Learning Objectives:**
- Compute distance and contact matrices
- Calculate backbone dihedral angles
- Understand SE(3) frame representations

In [ ]:
# !pip install biotite py3Dmol -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from biotite.database import rcsb
import biotite.structure as struc
import biotite.structure.io.pdb as pdb

np.random.seed(42)

## 1. Load a Protein Structure

In [ ]:
# Download ubiquitin
pdb_path = rcsb.fetch('1UBQ', 'pdb', target_path='.')
pdb_file = pdb.PDBFile.read(pdb_path)
structure = pdb_file.get_structure(model=1)
protein = structure[struc.filter_amino_acids(structure)]

# Extract CA coordinates
ca_mask = protein.atom_name == 'CA'
ca_coords = protein.coord[ca_mask]

print(f"Loaded structure with {len(ca_coords)} residues")

## 2. Distance Matrix

In [ ]:
def compute_distance_matrix(coords):
    """Compute pairwise Euclidean distance matrix."""
    diff = coords[:, np.newaxis, :] - coords[np.newaxis, :, :]
    return np.sqrt(np.sum(diff ** 2, axis=-1))

dist_matrix = compute_distance_matrix(ca_coords)
print(f"Distance matrix shape: {dist_matrix.shape}")
print(f"Distance range: {dist_matrix.min():.1f} - {dist_matrix.max():.1f} Å")

In [ ]:
# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

im1 = axes[0].imshow(dist_matrix, cmap='viridis', origin='lower')
axes[0].set_xlabel('Residue')
axes[0].set_ylabel('Residue')
axes[0].set_title('CA-CA Distance Matrix')
plt.colorbar(im1, ax=axes[0], label='Distance (Å)')

# Histogram of distances
triu_idx = np.triu_indices(len(dist_matrix), k=1)
distances = dist_matrix[triu_idx]
axes[1].hist(distances, bins=50, edgecolor='black', alpha=0.7)
axes[1].axvline(x=8.0, color='r', linestyle='--', label='8Å threshold')
axes[1].set_xlabel('Distance (Å)')
axes[1].set_ylabel('Count')
axes[1].set_title('Pairwise Distance Distribution')
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Contact Map

In [ ]:
def compute_contact_map(coords, threshold=8.0):
    """Compute binary contact map."""
    dist_matrix = compute_distance_matrix(coords)
    return (dist_matrix < threshold).astype(np.float32)

contact_map = compute_contact_map(ca_coords, threshold=8.0)
n_contacts = (contact_map.sum() - len(contact_map)) / 2
print(f"Number of contacts: {int(n_contacts)}")

In [ ]:
# Analyze contacts by sequence separation
n = len(contact_map)
local_mask = np.abs(np.arange(n)[:,None] - np.arange(n)[None,:]) < 6
medium_mask = (np.abs(np.arange(n)[:,None] - np.arange(n)[None,:]) >= 6) & \
              (np.abs(np.arange(n)[:,None] - np.arange(n)[None,:]) < 12)
long_mask = np.abs(np.arange(n)[:,None] - np.arange(n)[None,:]) >= 12

local_contacts = (contact_map * local_mask).sum() / 2
medium_contacts = (contact_map * medium_mask).sum() / 2
long_contacts = (contact_map * long_mask).sum() / 2

print(f"Contact breakdown:")
print(f"  Local (|i-j| < 6):     {int(local_contacts)}")
print(f"  Medium (6 <= |i-j| < 12): {int(medium_contacts)}")
print(f"  Long-range (|i-j| >= 12): {int(long_contacts)}")

In [ ]:
# Visualize with separation coloring
fig, ax = plt.subplots(figsize=(8, 8))

# Color different contact types
colored = np.zeros((n, n, 3))
colored[contact_map.astype(bool) & local_mask] = [0, 0, 1]   # Blue: local
colored[contact_map.astype(bool) & medium_mask] = [0, 1, 0]  # Green: medium
colored[contact_map.astype(bool) & long_mask] = [1, 0, 0]    # Red: long-range

ax.imshow(colored, origin='lower')
ax.set_xlabel('Residue')
ax.set_ylabel('Residue')
ax.set_title('Contact Map (Blue: local, Green: medium, Red: long-range)')
plt.show()

## 4. Backbone Dihedral Angles

In [ ]:
def compute_dihedral(p1, p2, p3, p4):
    """Compute dihedral angle between four points."""
    b1 = p2 - p1
    b2 = p3 - p2
    b3 = p4 - p3
    
    n1 = np.cross(b1, b2)
    n2 = np.cross(b2, b3)
    
    n1 = n1 / (np.linalg.norm(n1) + 1e-10)
    n2 = n2 / (np.linalg.norm(n2) + 1e-10)
    
    m1 = np.cross(n1, b2 / (np.linalg.norm(b2) + 1e-10))
    
    return np.arctan2(np.dot(m1, n2), np.dot(n1, n2))

# Extract backbone atoms
def get_backbone(structure):
    """Extract N, CA, C, O coordinates for each residue."""
    protein = structure[struc.filter_amino_acids(structure)]
    residue_ids = struc.get_residues(protein)[0]
    
    backbone_atoms = ['N', 'CA', 'C', 'O']
    coords = np.zeros((len(residue_ids), 4, 3))
    
    for i, res_id in enumerate(residue_ids):
        res_mask = protein.res_id == res_id
        res_atoms = protein[res_mask]
        for j, atom in enumerate(backbone_atoms):
            atom_mask = res_atoms.atom_name == atom
            if np.any(atom_mask):
                coords[i, j] = res_atoms.coord[atom_mask][0]
    
    return coords

backbone = get_backbone(structure)
print(f"Backbone shape: {backbone.shape}")

In [ ]:
# Compute phi and psi
n_res = len(backbone)
phi = np.full(n_res, np.nan)
psi = np.full(n_res, np.nan)

for i in range(n_res):
    if i > 0:
        phi[i] = compute_dihedral(
            backbone[i-1, 2], backbone[i, 0], backbone[i, 1], backbone[i, 2]
        )
    if i < n_res - 1:
        psi[i] = compute_dihedral(
            backbone[i, 0], backbone[i, 1], backbone[i, 2], backbone[i+1, 0]
        )

phi_deg = np.degrees(phi)
psi_deg = np.degrees(psi)

# Ramachandran plot
plt.figure(figsize=(8, 8))
valid = ~(np.isnan(phi_deg) | np.isnan(psi_deg))
plt.scatter(phi_deg[valid], psi_deg[valid], alpha=0.7, s=50)
plt.xlabel('Phi (degrees)')
plt.ylabel('Psi (degrees)')
plt.title('Ramachandran Plot - Ubiquitin')
plt.xlim(-180, 180)
plt.ylim(-180, 180)
plt.grid(True, alpha=0.3)
plt.axhline(0, color='gray', linewidth=0.5)
plt.axvline(0, color='gray', linewidth=0.5)
plt.show()

## 5. Sin/Cos Encoding for Angles

In [ ]:
def encode_dihedrals(phi, psi):
    """
    Encode dihedral angles using sin/cos.
    This provides a continuous representation where -180° ≈ 180°.
    """
    encoding = np.stack([
        np.sin(phi), np.cos(phi),
        np.sin(psi), np.cos(psi)
    ], axis=-1)
    return np.nan_to_num(encoding, 0.0)

dihedral_encoding = encode_dihedrals(phi, psi)
print(f"Dihedral encoding shape: {dihedral_encoding.shape}")
print(f"\nFirst 5 residues:")
print(dihedral_encoding[:5])

In [ ]:
# Visualize encoding along sequence
fig, axes = plt.subplots(4, 1, figsize=(14, 8), sharex=True)
labels = ['sin(φ)', 'cos(φ)', 'sin(ψ)', 'cos(ψ)']

for i, (ax, label) in enumerate(zip(axes, labels)):
    ax.plot(dihedral_encoding[:, i], 'b-', linewidth=1)
    ax.set_ylabel(label)
    ax.set_ylim(-1.1, 1.1)
    ax.axhline(0, color='gray', linewidth=0.5)

axes[-1].set_xlabel('Residue Position')
fig.suptitle('Dihedral Angle Encoding', y=1.02)
plt.tight_layout()
plt.show()

## 6. Local Coordinate Frames (SE(3))

In [ ]:
def compute_local_frame(n, ca, c):
    """
    Compute local coordinate frame for a residue.
    
    Returns:
        R: (3, 3) rotation matrix (local -> global)
        t: (3,) translation (CA position)
    """
    # X-axis: CA -> C direction
    x = c - ca
    x = x / np.linalg.norm(x)
    
    # Y-axis: perpendicular in N-CA-C plane
    n_vec = n - ca
    y = n_vec - np.dot(n_vec, x) * x
    y = y / np.linalg.norm(y)
    
    # Z-axis: cross product
    z = np.cross(x, y)
    
    R = np.column_stack([x, y, z])
    t = ca
    
    return R, t

# Compute frames for all residues
rotations = []
translations = []

for i in range(len(backbone)):
    R, t = compute_local_frame(backbone[i, 0], backbone[i, 1], backbone[i, 2])
    rotations.append(R)
    translations.append(t)

rotations = np.array(rotations)
translations = np.array(translations)

print(f"Rotations shape: {rotations.shape}")
print(f"Translations shape: {translations.shape}")

In [ ]:
# Verify rotation matrices are valid (orthogonal, det=1)
for i in range(5):
    R = rotations[i]
    det = np.linalg.det(R)
    orth_check = np.allclose(R @ R.T, np.eye(3))
    print(f"Residue {i+1}: det(R)={det:.4f}, orthogonal={orth_check}")

In [ ]:
# Relative transformations between consecutive residues
def compute_relative_transform(R1, t1, R2, t2):
    """Compute transform from frame 1 to frame 2."""
    R_rel = R1.T @ R2  # Relative rotation
    t_rel = R1.T @ (t2 - t1)  # Relative translation in frame 1
    return R_rel, t_rel

# Compute for consecutive residues
for i in range(3):
    R_rel, t_rel = compute_relative_transform(
        rotations[i], translations[i],
        rotations[i+1], translations[i+1]
    )
    print(f"Residue {i+1} -> {i+2}:")
    print(f"  Translation: {t_rel}")
    print(f"  Translation distance: {np.linalg.norm(t_rel):.2f} Å")

## Summary

| Representation | Shape | Key Properties |
|----------------|-------|----------------|
| Distance Matrix | (N, N) | Symmetric, rotation invariant |
| Contact Map | (N, N) | Binary, sparse for large proteins |
| Dihedral Angles | (N, 4) | Sin/cos encoding for continuity |
| SE(3) Frames | (N, 3, 3) + (N, 3) | Full rigid body representation |

**Key insights:**
- Distance matrices are rotation/translation invariant
- Contact maps are sparse representations of structure
- Dihedral angles capture local backbone geometry
- SE(3) frames are used in modern structure prediction (AlphaFold, RFDiffusion)